# LeetCode #1279: Traffic Light Controlled Intersection

https://leetcode.com/problems/traffic-light-controlled-intersection/

## Synchronization Approaches

| Approach | Mechanism | Notes |
| :--- | :--- | :--- |
| **Naive: Busy-Wait** | Spin checking green road | Wastes CPU; unsafe race on `greenRoad` |
| **Optimal: Mutex/Lock for Shared State ★** | `lock` + `SemaphoreSlim(1,1)` | Serializes cars; ensures only one road is green at a time |

---

## Understanding the Methods

### Naive: Busy-Wait
Spin in a loop checking a shared flag. Works but burns CPU.

### Optimal: Mutex/Lock for Shared State ★
A single `SemaphoreSlim(1,1)` (mutex) serializes all calls to `carArrived`. The first car of a new road sets `turnGreen` to switch the light; subsequent cars on the same road pass without switching. The mutex prevents two cars from simultaneously reading and writing `greenRoad`.

**Constraints:**
* `carId` and `roadId` identify each car and which road it uses (1 or 2)
* `turnGreen()` switches the traffic light
* `crossCar()` lets the car pass
* Only one car may be processed at a time


## Solutions
### C#

In [ ]:
using System.Threading;

public class TrafficLight
{
    private int _greenRoad = 1;  // road currently green
    private readonly SemaphoreSlim _mutex = new SemaphoreSlim(1, 1);

    public void CarArrived(
        int carId,
        int roadId,
        int direction,
        Action turnGreen,
        Action crossCar)
    {
        _mutex.Wait();
        try
        {
            if (_greenRoad != roadId)
            {
                turnGreen();
                _greenRoad = roadId;
            }
            crossCar();
        }
        finally
        {
            _mutex.Release();
        }
    }
}

### Python

In [ ]:
import threading

class TrafficLight:
    def __init__(self):
        self.green_road = 1
        self.lock = threading.Lock()

    def carArrived(
        self,
        carId: int,
        roadId: int,
        direction: int,
        turnGreen: 'Callable[[], None]',
        crossCar: 'Callable[[], None]'
    ) -> None:
        with self.lock:
            if self.green_road != roadId:
                turnGreen()
                self.green_road = roadId
            crossCar()

### Go

In [ ]:
package main

import "sync"

type TrafficLight struct {
	mu        sync.Mutex
	greenRoad int
}

func NewTrafficLight() *TrafficLight {
	return &TrafficLight{greenRoad: 1}
}

func (t *TrafficLight) CarArrived(
	carId, roadId, direction int,
	turnGreen func(),
	crossCar func(),
) {
	t.mu.Lock()
	defer t.mu.Unlock()
	if t.greenRoad != roadId {
		turnGreen()
		t.greenRoad = roadId
	}
	crossCar()
}

### Rust

In [ ]:
use std::sync::{Arc, Mutex};

struct TrafficLight {
    green_road: Mutex<i32>,
}

impl TrafficLight {
    fn new() -> Arc<Self> {
        Arc::new(TrafficLight { green_road: Mutex::new(1) })
    }

    fn car_arrived(
        &self,
        _car_id: i32,
        road_id: i32,
        _direction: i32,
        turn_green: impl Fn(),
        cross_car: impl Fn(),
    ) {
        let mut green = self.green_road.lock().unwrap();
        if *green != road_id {
            turn_green();
            *green = road_id;
        }
        cross_car();
    }
}

## Concurrency Scenarios

1. **Two cars on road 1 arrive sequentially**: First car acquires mutex, road 1 is already green, no light change; second car does the same — both cross without calling `turnGreen`.
2. **Car on road 2 arrives while road 1 is green**: Car 2 acquires mutex, sees `greenRoad == 1`, calls `turnGreen`, updates `greenRoad = 2`, then crosses.
3. **Simultaneous arrival from both roads**: One car acquires the mutex first; the other waits. The waiting car then checks the (possibly updated) green road state.
4. **Alternating roads**: Road 1 car, road 2 car, road 1 car — `turnGreen` called exactly twice; mutex ensures they never overlap.
5. **Many cars, same road**: All queue on the mutex; first car may switch the light; all subsequent same-road cars skip `turnGreen` — O(1) per car.
